# Integração do lab via Spark

Lê o que os outros serviços produziram no exemplo `lab_end_to_end`:

| Origem | Dado | Onde |
|---|---|---|
| NiFi | eventos JSON brutos | MinIO `s3a://bronze/nifi/events/` |
| NiFi + Airflow | eventos | Kafka `lab_events` |
| Spark (DAG) | tabela Iceberg | `iceberg.lab.events` (Hive Metastore) |
| Spark (DAG) | tabela Delta | `spark_catalog.lab.events_summary` |
| **Trino** (DAG) | tabela Iceberg criada via CTAS | `iceberg.lab.events_by_source` |

Rode antes o DAG `lab_end_to_end` no Airflow (http://localhost:8084).

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("lab_integracao")
         .master("spark://spark-master:7077")
         .getOrCreate())
print(spark.conf.get("spark.sql.defaultCatalog"))

## MinIO: eventos brutos gravados pelo NiFi

In [ ]:
raw = spark.read.option("recursiveFileLookup", "true").json("s3a://bronze/nifi/events/")
print("arquivos/eventos do NiFi:", raw.count())
raw.orderBy(F.desc("ts")).show(5, truncate=False)

## Kafka: tópico `lab_events` (NiFi + Airflow)

In [ ]:
kafka = (spark.read.format("kafka")
         .option("kafka.bootstrap.servers", "broker:19092")
         .option("subscribe", "lab_events")
         .option("startingOffsets", "earliest")
         .load())
(kafka.select(F.get_json_object(F.col("value").cast("string"), "$.source").alias("source"))
      .groupBy("source").count().show())

## Hive Metastore: tabelas Iceberg/Delta (Spark) e a tabela criada pelo Trino

In [ ]:
spark.sql("SHOW TABLES IN iceberg.lab").show()
spark.table("spark_catalog.lab.events_summary").orderBy("source", "event_type").show()
print("Criada pelo Trino:")
spark.table("iceberg.lab.events_by_source").show()

In [ ]:
# Conferência: o total do resumo (Delta) bate com a tabela de eventos (Iceberg)
total_events = spark.table("iceberg.lab.events").count()
total_summary = spark.table("spark_catalog.lab.events_summary").agg(F.sum("n_events")).first()[0]
print(total_events, total_summary)
assert total_events == total_summary

In [ ]:
spark.stop()